Session import and session buliding

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Cross_System_Monitoring")
    # delta lake configurations
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.file.impl", "org.apache.hadoop.fs.local.LocalFs")
    # network configurations to prevent Py4J Java timeouts
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

load delta table 

In [2]:
crm = spark.read.format("delta").load("../bronze/crm")

billing = spark.read.format("delta").load("../bronze/billing")

analytics = spark.read.format("delta").load("../bronze/analytics")

droping duplicates

In [3]:
crm = crm.dropDuplicates(["customer_id"])

billing = billing.dropDuplicates(["transaction_id"])

analytics = analytics.dropDuplicates()

In [4]:
crm = crm.na.drop(subset=["customer_id"])

billing = billing.na.drop(subset=["customer_id"])

changing column name

In [5]:
from pyspark.sql.functions import col

crm = crm.select(
    [col(c).alias(c.lower().replace(" ","_"))
     for c in crm.columns]
)

billing = billing.select(
    [col(c).alias(c.lower().replace(" ","_"))
     for c in billing.columns]
)

analytics = analytics.select(
    [col(c).alias(c.lower().replace(" ","_"))
     for c in analytics.columns]
)

saving process data into deelta table (silver)

In [6]:
crm.write.format("delta").mode("overwrite").save("../silver/crm")

billing.write.format("delta").mode("overwrite").save("../silver/billing")

analytics.write.format("delta").mode("overwrite").save("../silver/analytics")